In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

shipment_lifecycle_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/shipment_lifecycle/"
)

gold_delivery_performance_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "gold/delivery_performance/"
)


# ============================================================
# 2. LECTURE DU LIFECYCLE SILVER
# ============================================================

shipment_lifecycle = (
    spark.read
    .format("delta")
    .load(shipment_lifecycle_path)
)

print(
    "Nombre de shipments disponibles :",
    shipment_lifecycle.count()
)


# ============================================================
# 3. CONSTRUIRE LA TABLE GOLD
# ============================================================

gold_delivery_performance = (
    shipment_lifecycle

    .select(
        "shipment_id",
        "order_id",
        "carrier_id",
        "origin_city",
        "destination_city",
        "shipment_time",
        "expected_delivery_time",
        "delivered_at",
        "current_status",
        "is_delivered",
        "is_lifecycle_complete",
        "delivery_duration_hours",
        "delivery_delay_hours"
    )

    # --------------------------------------------------------
    # Classification livraison à l'heure / en retard
    # --------------------------------------------------------

    .withColumn(
        "is_late",

        F.when(
            F.col("delivered_at").isNull(),
            F.lit(None)
        )

        .otherwise(
            F.col("delivery_delay_hours") > 0
        )
    )

    .withColumn(
        "delivery_performance_status",

        F.when(
            F.col("delivered_at").isNull(),
            F.lit("NOT_DELIVERED")
        )

        .when(
            F.col("delivery_delay_hours") > 0,
            F.lit("LATE")
        )

        .otherwise(
            F.lit("ON_TIME")
        )
    )

    # --------------------------------------------------------
    # Date utile pour les analyses temporelles
    # --------------------------------------------------------

    .withColumn(
        "shipment_date",
        F.to_date(
            F.col("shipment_time")
        )
    )

    .withColumn(
        "delivery_date",
        F.to_date(
            F.col("delivered_at")
        )
    )
)


# ============================================================
# 4. ÉCRITURE GOLD DELTA
# ============================================================

(
    gold_delivery_performance
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_delivery_performance_path)
)

print(
    "gold_delivery_performance créée avec succès."
)


# ============================================================
# 5. VÉRIFICATION
# ============================================================

print(
    "Nombre de lignes Gold :",
    gold_delivery_performance.count()
)

display(
    gold_delivery_performance
    .orderBy("shipment_id")
)


# ============================================================
# 6. PREMIERS KPI
# ============================================================

print("=== KPI DELIVERY PERFORMANCE ===")

delivery_kpis = (
    gold_delivery_performance

    .agg(

        F.count("*")
        .alias("total_shipments"),

        F.sum(
            F.col("is_delivered")
            .cast("int")
        )
        .alias("delivered_shipments"),

        F.sum(
            F.when(
                F.col("is_late") == True,
                1
            ).otherwise(0)
        )
        .alias("late_shipments"),

        F.avg(
            "delivery_duration_hours"
        )
        .alias("avg_delivery_duration_hours"),

        F.avg(
            F.when(
                F.col("is_delivered") == True,
                (~F.col("is_late")).cast("int")
            )
        )
        .alias("on_time_delivery_rate")
    )

    .withColumn(
        "on_time_delivery_rate_pct",
        F.round(
            F.col("on_time_delivery_rate") * 100,
            2
        )
    )
)

display(delivery_kpis)